# multi_strat_052

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (88).ipynb`

| Field | Value |
|---|---|
| Section | `strategies` |
| Market | `unknown` |
| Trading style | `unknown` |
| Timeframe | `unknown` |
| Code cells | 12 |
| Detected functions | create_75min_candles, normalized_dema_strategy, ema, dema, process_ticker, backtest_trades, calculate_drawdown, evaluate_params, compute_norm_dema_for_group |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** _TODO_
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
data = pd.read_csv('/content/drive/MyDrive/5min_data.csv')

data

In [ ]:
data['Ticker'].unique()

In [ ]:
# Convert 'Date' column to datetime if it isn't already
data['Date'] = pd.to_datetime(data['Date'])

# Define the date range
start_date = '2023-01-01'
end_date = '2025-12-30'

# Define the tickers you want to filter
selected_tickers = ['TRENT', 'ADANIENT', 'ADANIPORTS', 'BAJAJ-AUTO', 'COALINDIA']  # Add tickers you want

# Apply filtering
data = data[
    (data["Date"] >= start_date) &
    (data["Date"] <= end_date) &
    (data["Ticker"].isin(selected_tickers))
]

# Display filtered results
data

In [ ]:
import pandas as pd

# Ensure datetime format
data['Date'] = pd.to_datetime(data['Date'])

# Add 'date_only' and 'time_only'
data['date_only'] = data['Date'].dt.date
data['time_only'] = data['Date'].dt.time

# Filter only within trading session
data = data[
    (data['time_only'] >= pd.to_datetime("09:15").time()) &
    (data['time_only'] <= pd.to_datetime("15:30").time())
]

# Drop temporary time column
data = data.drop(columns=['time_only'])

# Function for 75-minute candles
def create_75min_candles(group):
    ticker = group['Ticker'].iloc[0]
    date = group['date_only'].iloc[0]
    group = group.set_index('Date')

    bins = [
        pd.Timestamp(f"{date} 09:15"),
        pd.Timestamp(f"{date} 10:30"),
        pd.Timestamp(f"{date} 11:45"),
        pd.Timestamp(f"{date} 13:00"),
        pd.Timestamp(f"{date} 14:15"),
        pd.Timestamp(f"{date} 15:30")
    ]

    # Assign candle start time as label
    group['Candle'] = pd.cut(group.index, bins=bins, labels=bins[:-1])

    # Drop rows not in bins
    group = group.dropna(subset=['Candle'])

    # Aggregate into OHLCV
    agg = group.groupby('Candle').agg({
        'open': 'first',
        'high': 'max',
        'low': 'min',
        'close': 'last',
        'volume': 'sum'
    }).reset_index().rename(columns={'Candle': 'Date'})

    agg['Ticker'] = ticker
    return agg

# Apply to all ticker-date groups
resampled_75min = (
    data.groupby(['Ticker', 'date_only'])
        .apply(create_75min_candles)
        .reset_index(drop=True)
)

# Reorder columns
resampled_75min = resampled_75min[['Ticker', 'Date', 'open', 'high', 'low', 'close', 'volume']]

# Assign to data if desired
data = resampled_75min


In [ ]:
data.dropna()
data

In [ ]:
data.columns

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from joblib import Parallel, delayed

import pandas as pd

def normalized_dema_strategy(df, len_dema=20, base_len=21, Lu=70, Su=30, vol_window=20, vol_thresh=1.2):
    def ema(series, span):
        return series.ewm(span=span, adjust=False).mean()

    def dema(series, length):
        e1 = ema(series, length)
        e2 = ema(e1, length)
        return 2 * e1 - e2

    df = df.copy()

    # Calculate DEMA and Normalized Base
    dema_series = dema(df['close'], len_dema)
    base = dema_series.rolling(base_len).mean()
    sd = dema_series.rolling(base_len).std() * 2
    upper_sd = base + sd
    lower_sd = base - sd
    norm_base = 100 * (dema_series - lower_sd) / (upper_sd - lower_sd)

    # Volume condition
    avg_volume = df['volume'].rolling(vol_window).mean()
    vol_condition = df['volume'] > vol_thresh * avg_volume

    # Store indicators
    df['NormBase'] = norm_base
    df['BASE'] = base
    df['UpperSD'] = upper_sd
    df['LowerSD'] = lower_sd
    df['AvgVolume'] = avg_volume
    df['VolCondition'] = vol_condition

    # Signal logic with volume filter
    df['Long'] = (df['NormBase'] > Lu) & (df['close'] > df['UpperSD']) & vol_condition
    df['Short'] = (df['NormBase'] < Su) & (df['close'] < df['LowerSD']) & vol_condition

    df['Signal'] = 0
    df.loc[df['Long'] & ~df['Short'], 'Signal'] = 1
    df.loc[df['Short'], 'Signal'] = 2

    return df

# Function to process trades for a single ticker
def process_ticker(ticker, ticker_data, initial_capital, brokerage, slippage, tp_pct, sl_pct, end_time, stop_trading_time, start_trading_time):
    """Processes trades for a single ticker, applying entry/exit logic and calculating PnL."""
    open_positions = {}
    trades = []
    cumulative_pnl = 0

    for row in ticker_data:
        timestamp, close, open_price, high, low, signal = row["date"], row["close"], row["open"], row["high"], row["low"], row["Signal"]

        # Check for open positions and apply exit logic
        if ticker in open_positions:
            position = open_positions[ticker]
            entry_price, direction, qty = position["entry_price"], position["direction"], position["qty"]
            exit_price, exit_reason = None, None

            # Compute Take Profit (TP) & Stop Loss (SL) prices
            tp_price = entry_price * (1 + tp_pct) if direction == "long" else entry_price * (1 - tp_pct)
            sl_price = entry_price * (1 - sl_pct) if direction == "long" else entry_price * (1 + sl_pct)

            # Determine exit conditions
            if (direction == "long" and high >= tp_price) or (direction == "short" and low <= tp_price):
                exit_price, exit_reason = tp_price, "Take Profit"
            elif (direction == "long" and low <= sl_price) or (direction == "short" and high >= sl_price):
                exit_price, exit_reason = sl_price, "Stop Loss"
            elif timestamp.astype('datetime64[m]').astype(str)[-5:] >= end_time:
                exit_price, exit_reason = close, "EOD Close"

            if exit_price is not None:
                # Apply slippage
                exit_price *= (1 - slippage) if direction == "long" else (1 + slippage)

                # Calculate PnL including brokerage
                raw_pnl = (exit_price - entry_price) * qty if direction == "long" else (entry_price - exit_price) * qty
                pnl = raw_pnl - (brokerage * initial_capital * 2)
                cumulative_pnl += pnl

                # Store trade details
                trades.append([ticker, position["entry_time"], timestamp, entry_price, exit_price, pnl, exit_reason, cumulative_pnl, direction])
                del open_positions[ticker]
                continue

        # Open new position
        if ticker not in open_positions and start_trading_time < timestamp.astype('datetime64[m]').astype(str)[-5:] < stop_trading_time:
            if signal == 1:
                direction = "long"
            elif signal == 2:
                direction = "short"
            else:
                continue

            entry_price = open_price * (1 + slippage) if direction == "long" else open_price * (1 - slippage)
            qty = initial_capital / entry_price
            open_positions[ticker] = {"entry_time": timestamp, "entry_price": entry_price, "direction": direction, "qty": qty}

    return trades

# Main backtesting function
def backtest_trades(data):
    """Runs backtest for all tickers in the dataset using parallel processing."""

    # Trading parameters
    initial_capital = 100000  # ₹1 lakh per trade
    brokerage = 0.001  # 0.1% brokerage
    slippage = 0.0001  # 0.01% slippage
    tp_pct = 0.02  # 1.5% take-profit
    sl_pct = 0.001  # 0.5% stop-loss
    end_time = "14:15"
    stop_trading_time = "14:15"
    start_trading_time = "09:15"

    # Get unique tickers
    unique_tickers = data['Ticker'].unique()

    # Convert DataFrame to NumPy structured array for performance
    dtype = [
        ("Ticker", "U10"), ("date", "M8[ms]"), ("close", "f8"),
        ("open", "f8"), ("high", "f8"), ("low", "f8"), ("Signal", "i4")
    ]

    data_np = np.array(
        list(data[["Ticker", "Date", "close", "open", "high", "low", "Signal"]]
            .itertuples(index=False, name=None)),
        dtype=dtype
    )

    # Run backtest in parallel
    results = Parallel(n_jobs=-1)(delayed(process_ticker)(ticker, data_np[data_np['Ticker'] == ticker], initial_capital, brokerage, slippage, tp_pct, sl_pct, end_time, stop_trading_time, start_trading_time) for ticker in tqdm(unique_tickers))

    # Flatten results into DataFrame
    trades = [trade for result in results for trade in result]
    return pd.DataFrame(trades, columns=["Ticker", "Entry Time", "Exit Time", "Entry Price", "Exit Price", "PnL", "Exit Reason", "Cumulative PnL", "Direction"])

# === Main Execution ===

# Ensure 'Date' is datetime
data['Date'] = pd.to_datetime(data['Date'])

# Compute signals
data = data.groupby('Ticker', group_keys=False).apply(compute_normalized_dema)

# Shift signals to use next-bar execution
data['Signal'] = data.groupby('Ticker')['Signal'].shift(1)
data = data.dropna(subset=['Signal'])
data['Signal'] = data['Signal'].astype(int)

# Run backtest
tradebook = backtest_trades(data)

# Save results
tradebook.to_csv("/content/drive/MyDrive/tradebook_DEMA.csv", index=False)
tradebook


In [ ]:
from itertools import product
import numpy as np
import pandas as pd
from tqdm import tqdm

def calculate_drawdown(pnls):
    cumulative = np.cumsum(pnls)
    peak = np.maximum.accumulate(cumulative)
    drawdown = peak - cumulative
    return np.max(drawdown)

def evaluate_params(len_dema, base_len, Lu, Su, tp_pct, sl_pct, data):
    def compute_norm_dema_for_group(group):
        group = group.copy()
        dema_series = dema(group['close'], len_dema)
        base = dema_series.rolling(base_len).mean()
        sd = dema_series.rolling(base_len).std() * 2
        upper_sd = base + sd
        lower_sd = base - sd
        norm_base = 100 * (dema_series - lower_sd) / (upper_sd - lower_sd)

        group['NormBase'] = norm_base
        group['BASE'] = base
        group['UpperSD'] = upper_sd
        group['LowerSD'] = lower_sd

        group['Long'] = (group['NormBase'] > Lu) & (group['close'] > group['LowerSD'])
        group['Short'] = group['NormBase'] < Su

        group['Signal'] = 0
        group.loc[group['Long'] & ~group['Short'], 'Signal'] = 1
        group.loc[group['Short'], 'Signal'] = 2

        # Shift signal by 1 to avoid lookahead bias
        group['Signal'] = group['Signal'].shift(1)

        return group

    data_copy = data.copy()
    data_copy = data_copy.groupby('Ticker', group_keys=False).apply(compute_norm_dema_for_group)
    data_copy = data_copy.dropna(subset=['Signal'])
    data_copy['Signal'] = data_copy['Signal'].astype(int)

    tradebook = backtest_trades(data_copy)

    if tradebook.empty:
        return {"params": (len_dema, base_len, Lu, Su, tp_pct, sl_pct), "pnl": 0, "dd": np.inf, "ratio": 0}

    total_profit = tradebook["PnL"].sum()
    max_drawdown = calculate_drawdown(tradebook["PnL"].values)
    ratio = total_profit / max_drawdown if max_drawdown > 0 else np.inf

    return {
        "params": (len_dema, base_len, Lu, Su, tp_pct, sl_pct),
        "pnl": total_profit,
        "dd": max_drawdown,
        "ratio": ratio
    }

# === Grid Search Ranges ===
len_dema_range = [5, 10, 15]
base_len_range = [5, 10, 15]
Lu_range = [85, 90, 95]
Su_range = [5, 10, 15]
tp_range = [0.02, 0.03]
sl_range = [0.001, 0.005, 0.01]

# === Run Grid Search ===
results = []
for len_dema, base_len, Lu, Su, tp_pct, sl_pct in tqdm(
    product(len_dema_range, base_len_range, Lu_range, Su_range, tp_range, sl_range)
):
    result = evaluate_params(len_dema, base_len, Lu, Su, tp_pct, sl_pct, data)
    results.append(result)
    print(f"Params: {result['params']} | PnL: {result['pnl']:.2f} | Max DD: {result['dd']:.2f} | PnL/DD: {result['ratio']:.2f}")

# === Sort and Display Best Result by Profit-to-Drawdown Ratio ===
results_sorted_by_ratio = sorted(results, key=lambda x: x["ratio"], reverse=True)
best_by_ratio = results_sorted_by_ratio[0]

# === Sort and Display Best Result by Total Profit ===
results_sorted_by_profit = sorted(results, key=lambda x: x["pnl"], reverse=True)
best_by_profit = results_sorted_by_profit[0]

print("\nBest Parameters by Profit-to-Drawdown Ratio:")
print(f"Params: {best_by_ratio['params']}")
print(f"Total Profit: {best_by_ratio['pnl']:.2f}")
print(f"Max Drawdown: {best_by_ratio['dd']:.2f}")
print(f"Profit-to-Drawdown Ratio: {best_by_ratio['ratio']:.2f}")

print("\nBest Parameters by Total Profit:")
print(f"Params: {best_by_profit['params']}")
print(f"Total Profit: {best_by_profit['pnl']:.2f}")
print(f"Max Drawdown: {best_by_profit['dd']:.2f}")
print(f"Profit-to-Drawdown Ratio: {best_by_profit['ratio']:.2f}")

In [ ]:
# Convert the results to a DataFrame
results_df = pd.DataFrame(results)

# Expand the parameter tuple into separate columns
results_df[['len_dema', 'base_len', 'Lu', 'Su', 'tp_pct', 'sl_pct']] = pd.DataFrame(
    results_df['params'].tolist(), index=results_df.index
)

# Drop the original tuple column
results_df = results_df.drop(columns=['params'])

# Reorder columns for readability
results_df = results_df[[
    'len_dema', 'base_len', 'Lu', 'Su', 'tp_pct', 'sl_pct',
    'pnl', 'dd', 'ratio'
]]

# Save to CSV
results_df.to_csv("/content/drive/MyDrive/grid_search_dema_results.csv", index=False)

print("Grid search results saved to CSV.")
